# 03. RAG 직접 구현 — 연세대학교 대학원 학칙 Q&A
**Day 4**

> 수업용 문서: **`samples/` 안의 연세대학교 대학원 학칙 PDF**

**핵심 데모:**
- RAG **없이** 질문 → 모델이 학칙 세부 조항을 **모름** (환각 가능)
- RAG **있으면** → 학칙 PDF에서 검색한 근거로 **정확히** 답변

---
## 0. 설치

In [11]:
!pip install openai python-dotenv pymupdf

In [12]:
import json, os, re, sys
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

BASE = Path('.').resolve()
sys.path.insert(0, str(BASE))
load_dotenv("../../.env")
api_key = os.getenv("OPENAI_KEY")
client = OpenAI(api_key=api_key)

In [13]:
pdf_path = os.path.join(os.getcwd(),'samples/학칙.pdf')
pdf_path

'/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/학칙.pdf'

In [14]:
import os
print(os.path.exists(pdf_path))

True


---
## 1. RAG가 필요한 이유 — 학칙 질문으로 확인


In [15]:
demo_question = '연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?'

plain = client.chat.completions.create(
    model='gpt-4o-mini',
    messages=[{'role': 'user', 'content': demo_question}],
).choices[0].message.content

print('질문:', demo_question)
print('\n=== RAG 없이 (프롬프트만) ===')
print(plain)


질문: 연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?

=== RAG 없이 (프롬프트만) ===
연세대학교에서 논문 제출 시한을 연장한 학생은 일반적으로 휴학이 가능합니다. 하지만 휴학 관련 규정은 학과나 전공에 따라 다를 수 있으며, 각 대학원별로 상이한 예외 사항이나 조건이 있을 수 있습니다.

일반적으로 논문 제출과 관련된 연장은 학생의 연구나 작성 상황을 반영하여 부여되는 것이기 때문에, 휴학을 신청할 경우에는 해당 학과의 규정을 체크하고, 필요할 경우 지도 교수나 학과 조교와 상담하는 것이 좋습니다.

예외로는, 특정 기간 내에 휴학을 하지 않을 경우 졸업이 불가능해지는 경우, 학점 요건을 충족해야 하는 경우 등이 있을 수 있습니다. 이는 각 학부나 대학원 규정에 따라 다를 수 있으므로, 반드시 해당 부서에 직접 문의하는 것이 가장 정확합니다.


---
## 2. Step 1 — 학칙 PDF → 텍스트 추출


In [16]:
print(pdf_path)

/Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/학칙.pdf


In [17]:
from pdf_summary import summarize_pdf

text_path = summarize_pdf(pdf_path)
print('text_path:',text_path)
with open(text_path, 'r', encoding = 'utf-8') as f:
    rules_text = f.read()
print('학칙 글자 수:', len(rules_text))


입력받은 pdf_path = /Users/seorincho/Desktop/code/2026_AI/SK Autonomous R&D/4일차/samples/학칙.pdf
.env loaded: /Users/seorincho/Desktop/code/2026_AI/.env
text_path: # 대학원 학칙

## 저자의 문제 인식 및 주장
대학원 학칙은 기독교 정신을 바탕으로 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여하는 것을 목적으로 한다. 학칙은 석사 및 박사학위 과정, 통합과정, 협동과정 등 다양한 학위과정을 규정하고 있으며, 각 과정의 수업연한과 재학연한을 명시하고 있다. 또한, 학생의 입학정원과 지원자격, 입학전형 방법 등을 상세히 규정하여 공정한 입학 절차를 보장하고 있다. 등록 및 학적 변동에 관한 규정은 학생의 권리와 의무를 명확히 하여 학업을 지속할 수 있는 환경을 조성한다. 학위논문 제출 자격시험과 수료 요건을 통해 학생의 학업 성취도를 평가하고, 학위 수여 및 취소에 대한 절차를 명확히 하여 학위의 신뢰성을 높이고 있다. 이러한 규정들은 대학원의 교육 품질을 유지하고, 학생들이 학문적 목표를 달성할 수 있도록 지원하는 데 중점을 두고 있다.

## 저자 소개
저자는 대학원 학칙을 제정하고 개정하는 과정에 참여한 교육 관계자들로, 대학원 교육의 질을 높이고 학생들의 학습 환경을 개선하기 위해 노력하고 있다. 이들은 기독교 정신을 바탕으로 한 교육 철학을 실현하기 위해 다양한 학문적 요구와 사회적 변화를 반영하여 학칙을 지속적으로 발전시키고 있다.


OSError: [Errno 63] File name too long: '# 대학원 학칙\n\n## 저자의 문제 인식 및 주장\n대학원 학칙은 기독교 정신을 바탕으로 창의적 이론과 과학적 방법을 탐구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여하는 것을 목적으로 한다. 학칙은 석사 및 박사학위 과정, 통합과정, 협동과정 등 다양한 학위과정을 규정하고 있으며, 각 과정의 수업연한과 재학연한을 명시하고 있다. 또한, 학생의 입학정원과 지원자격, 입학전형 방법 등을 상세히 규정하여 공정한 입학 절차를 보장하고 있다. 등록 및 학적 변동에 관한 규정은 학생의 권리와 의무를 명확히 하여 학업을 지속할 수 있는 환경을 조성한다. 학위논문 제출 자격시험과 수료 요건을 통해 학생의 학업 성취도를 평가하고, 학위 수여 및 취소에 대한 절차를 명확히 하여 학위의 신뢰성을 높이고 있다. 이러한 규정들은 대학원의 교육 품질을 유지하고, 학생들이 학문적 목표를 달성할 수 있도록 지원하는 데 중점을 두고 있다.\n\n## 저자 소개\n저자는 대학원 학칙을 제정하고 개정하는 과정에 참여한 교육 관계자들로, 대학원 교육의 질을 높이고 학생들의 학습 환경을 개선하기 위해 노력하고 있다. 이들은 기독교 정신을 바탕으로 한 교육 철학을 실현하기 위해 다양한 학문적 요구와 사회적 변화를 반영하여 학칙을 지속적으로 발전시키고 있다.'

In [ ]:
print(rules_text[rules_text.find('제2조의7'):rules_text.find('제2조의7')+350])

제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.
1. 석사학위과정: 2년(4학기)
2. 박사학위과정: 2년(4학기)
3. 통합과정: 3년(6학기)
② 제1항의 규정에도 불구하고 학위수여 자격 요건을 충족한 학생은 수업연한을 1
학기 단축할 수 있다. 단, 통합과정 및 석사학위논문 대체실적을 통해 학위를 취득
하고자 하는 학생은 제외한다.
제2조의8(재학연한) ① 대학원 각 학위과정의 재학연한은 다음 각호와 같으며, 이를 초
과할 수 없다. 단, 휴학 및 제적 기간은 재학연한에 산입하지 아니한다. 
1. 석사학위과정: 4년(8학기)
2. 박사학위과정: 7년(14학기)
3. 통합과정: 8년(16


---
## 3. Step 2 — 청크(Chunk) 분할

학칙 전문은 약 4만 자 이상 → 검색 가능한 크기로 나눕니다.

In [ ]:
rules_text

'Ⅰ. 대학원 학칙  /  7\nⅠ. 대학원 학칙\n제정: 1974. 05. 18.\n개정(제122차): 2026. 06. 06.\n제1장 총칙\n제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐\n구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다.\n제2장 과정 및 정원\n제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 \n위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사\n학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, \n학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다.\n제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 \n공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또\n는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동\n과정”이라 한다)을 둘 수 있다.\n② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 \n의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다.\n③ <삭제> \n제2조의3 <삭제>\n제2조의4 <삭제>\n제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와 대학원을 연계하는 학부\n-대학원 연계과정을 둔다. 연계과정 운영에 관한 사항은 따로 정한다. \n제2조의6(계약학과) ① 대학원에 두는 학위과정으로 국가, 지방자치단체 또는 산업체 \n등과의 계약에 의한 학과를 둘 수 있으며, 이의 운영에 관한 사항은 따로 정한다.\n② 대학원장은 대학원운영위원회 의결을 거쳐 계약학과의 존속 여부 등을 결정하며, \n이에 관한 사항은 따로 정한다.\n\n-------------------\n8\n③ <삭제> \n제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다.\n1. 석사학위과정: 2년(4학기)\n

In [ ]:
## 앞뒤 빈 줄 공백 제거
## 연속된 공백 문자를 공백 하나로 바꿈
text = re.sub(r'\s+', ' ', rules_text.strip())

In [ ]:
text

'Ⅰ. 대학원 학칙 / 7 Ⅰ. 대학원 학칙 제정: 1974. 05. 18. 개정(제122차): 2026. 06. 06. 제1장 총칙 제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐 구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다. 제2장 과정 및 정원 제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사 학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다. 제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또 는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동 과정”이라 한다)을 둘 수 있다. ② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다. ③ <삭제> 제2조의3 <삭제> 제2조의4 <삭제> 제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와 대학원을 연계하는 학부 -대학원 연계과정을 둔다. 연계과정 운영에 관한 사항은 따로 정한다. 제2조의6(계약학과) ① 대학원에 두는 학위과정으로 국가, 지방자치단체 또는 산업체 등과의 계약에 의한 학과를 둘 수 있으며, 이의 운영에 관한 사항은 따로 정한다. ② 대학원장은 대학원운영위원회 의결을 거쳐 계약학과의 존속 여부 등을 결정하며, 이에 관한 사항은 따로 정한다. ------------------- 8 ③ <삭제> 제2조의7(수업연한) ① 대학원 각 학위과정의 수업연한은 다음 각호와 같다. 1. 석사학위과정: 2년(4학기) 2. 박사학위과정: 2년(4학기) 3. 통합과정: 3년(6학기) ② 제1항의 규정

In [ ]:
def split_text(text, chunk_size=280, overlap=60):
    text = re.sub(r'\s+', ' ', text.strip())
    if len(text) <= chunk_size:
        return [text]
    chunks, start = [], 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return chunks

chunks = split_text(rules_text)
print('청크 수:', len(chunks))
print('[C001]', chunks[0][:150], '...')

청크 수: 212
[C001] Ⅰ. 대학원 학칙 / 7 Ⅰ. 대학원 학칙 제정: 1974. 05. 18. 개정(제122차): 2026. 06. 06. 제1장 총칙 제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐 구하고, 지도적 인격을 도야하여 인류 문화 향상에 기 ...


---
## 4. Step 3 — 인덱스 구축

In [ ]:
INDEX = [
    {
        'chunk_id': f'RULE_C{i:03d}',
        'title': '연세대학교 대학원 학칙',
        'text': chunk,
    }
    for i, chunk in enumerate(chunks, 1)
]
print('인덱스 청크:', len(INDEX))
print(INDEX)

인덱스 청크: 212
[{'chunk_id': 'RULE_C001', 'title': '연세대학교 대학원 학칙', 'text': 'Ⅰ. 대학원 학칙 / 7 Ⅰ. 대학원 학칙 제정: 1974. 05. 18. 개정(제122차): 2026. 06. 06. 제1장 총칙 제1조(목적) 대학원은 기독교 정신을 바탕으로 하여 창의적 이론과 과학적 방법을 탐 구하고, 지도적 인격을 도야하여 인류 문화 향상에 기여함을 목적으로 한다. 제2장 과정 및 정원 제2조(과정) 대학원에는 석사학위를 취득하기 위한 석사학위과정, 박사학위를 취득하기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사 학위과정과 박사학위과정이 '}, {'chunk_id': 'RULE_C002', 'title': '연세대학교 대학원 학칙', 'text': '기 위한 박사학위과정, 석사학위과정을 거치지 않고 박사학위를 취득하기 위하여 석사 학위과정과 박사학위과정이 통합된 과정(석·박사 통합과정, 이하 “통합과정”)을 두며, 학위과정 외에 학위를 수여하지 아니하는 연구과정을 둘 수 있다. 제2조의2(협동과정) ① 대학원에 두는 학위과정으로 학과 외에 두 개 이상의 학과가 공동으로 설치·운영하는 협동과정(이하 “학과 간 협동과정”이라 한다)과 연구기관 또 는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동 과정”이라 한다'}, {'chunk_id': 'RULE_C003', 'title': '연세대학교 대학원 학칙', 'text': '관 또 는 산업체와의 계약에 의하여 설치·운영하는 학·연·산 협동과정(이하 “학·연·산 협동 과정”이라 한다)을 둘 수 있다. ② 대학원장은 학과 간 협동과정의 운영실적을 2년마다 평가하여 대학원운영위원회 의결을 거쳐 존속여부를 결정하며, 이에 관한 사항은 따로 정한다. ③ <삭제> 제2조의3 <삭제> 제2조의4 <삭제> 제2조의5(연계과정) 대학원에 두는 학위과정으로 본교 학부와 대학원을 연계하는 학부 -대학원 연계과정을 

---
## 5. Step 4 — Retrieval (검색)

수업용: 질문 키워드와 청크 키워드 **겹침 개수**로 순위를 매깁니다.

In [ ]:
def tokenize(text):
    return {t.lower() for t in re.findall(r'[가-힣a-zA-Z0-9]+', text) if len(t) > 1}

def search(query, top_k=3):
    q = tokenize(query)
    scored = []
    for item in INDEX:
        score = len(q & tokenize(item['text']))
        if score > 0:
            scored.append((score, item))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [{'score': s, **it} for s, it in scored[:top_k]]

hits = search(demo_question, top_k=3)
for h in hits:
    print(f"[{h['chunk_id']}] score={h['score']}")
    print(h['text'][:200], '...\n')

[RULE_C039] score=4
, 휴학 및 제적기간은 산입하지 아니한다. ③ 수료요건을 충족하고 합당한 사유가 있는 학생은 논문제출 시한 연장 사유서와 연구계획서를 제출하고 지도교수와 대학원장의 승인을 얻어 논문제출 시한을 2년 연 장할 수 있다. ④ 논문제출 시한 연장을 승인받은 기간에는 휴학할 수 없으나 다음 각호의 휴학은 예외로 처리할 수 있다. 1. 입대휴학 2. 육아휴학 3.  ...

[RULE_C040] score=3
사유에 의한 일반휴학 ⑤ 「장애인 등에 대한 특수교육법」 제15조 제1항의 특수교육대상자는 「대학원 학 칙」 제2조의8 제2항에 따라 합당한 사유가 있는 경우 대학원장의 승인을 받아 제3 항의 연장 시한을 예외로 한다. 제36조의2(논문제출) 학위논문 인준을 받은 학생은 대학원에서 정한 절차에 따라 완성 된 학위논문을 제출하여야 한다. 제8장 연구과정생,  ...

[RULE_C038] score=2
절한 수정 보완을 하여 재심사를 받을 수 있으며 이에 관한 사항은 따로 정한다. ② <삭제> 제36조(논문제출 시한) ① 석사학위과정 학생은 입학일로부터 4년 이내에, 박사학위과 정 학생은 입학일로부터 7년 이내에, 통합과정 학생은 입학일로부터 8년 이내에 학 위논문 심사에 합격하여 완성된 학위논문을 대학원에서 정한 절차에 따라 제출하여 야 한다. ② 위  ...



---
## 6. Step 5 — Generation (근거 기반 답변)

In [ ]:
demo_question

'연세대학교에서 논문제출 시한을 연장한 학생은 휴학할 수 있는가? 예외는 무엇인가?'

In [ ]:
def answer_with_rag(question, top_k=3):
    hits = search(question, top_k=top_k)
    if not hits:
        return {'question': question, 'answer': '관련 학칙 조항을 찾지 못했습니다.', 'sources': []}
    context = '\n\n'.join(f"[{h['chunk_id']}] {h['text']}" for h in hits)
    print("context:",context)
    r = client.chat.completions.create(
        model='gpt-4o-mini', temperature=0.1,
        messages=[
            {'role': 'system', 'content': 'Use only provided context from university regulations. Answer in Korean.'},
            {'role': 'user', 'content': f'질문: {question}\n\n학칙 문맥:\n{context}\n\n문맥에 없으면 모른다고 답하고 chunk_id를 bullet로 적으세요.'},
        ],
    )
    return {
        'question': question,
        'answer': r.choices[0].message.content,
        'sources': [h['chunk_id'] for h in hits],
    }

rag_result = answer_with_rag(demo_question)
print('=== RAG 사용 ===')
print(rag_result['answer'])
print('\n출처:', rag_result['sources'])

NameError: name 'search' is not defined

---
## 7. RAG vs 비-RAG — 학칙 질문 3개 비교

In [ ]:
questions = [
    '연세대학교 대학원 석사학위과정 수업연한은?',
    '박사학위과정 재학연한은 몇 년인가요?',
    '통합과정 재학연한은 몇 년(몇 학기)인가요?',
]

for q in questions:
    plain = client.chat.completions.create(
        model='gpt-4o-mini', messages=[{'role': 'user', 'content': q}]
    ).choices[0].message.content
    rag = answer_with_rag(q)['answer']
    print('\n' + '='*60)
    print('Q:', q)
    print('[RAG 없음]', plain[:180])
    print('[RAG 있음]', rag[:280])


Q: 연세대학교 대학원 석사학위과정 수업연한은?
[RAG 없음] 연세대학교 대학원 석사학위과정의 일반적인 수업연한은 보통 2년입니다. 그러나 전공이나 프로그램에 따라 다소 차이가 있을 수 있으니, 구체적인 정보는 연세대학교 대학원 공식 웹사이트나 해당 학과의 안내를 참고하는 것이 좋습니다.
[RAG 있음] 연세대학교 대학원 석사학위과정의 수업연한은 2년(4학기)입니다. 

- chunk_id: RULE_C005

Q: 박사학위과정 재학연한은 몇 년인가요?
[RAG 없음] 박사학위 과정의 재학연한은 국가, 대학, 전공 분야에 따라 다를 수 있지만, 일반적으로 3년에서 5년 정도입니다. 일부 프로그램에서는 석사학위 소지자는 더 단축된 기간 동안 연구를 수행할 수 있습니다. 더 구체적인 정보를 얻기 위해서는 해당 대학의 규정을 참조하시거나 관련 부서에 문의하는 것이 좋습니다.
[RAG 있음] 박사학위과정의 재학연한은 7년(14학기)입니다. 

- chunk_id: [RULE_C006]

Q: 통합과정 재학연한은 몇 년(몇 학기)인가요?
[RAG 없음] 통합과정의 재학연한은 일반적으로 3년에서 5년까지 다양합니다. 보통 석사와 박사를 통합하여 진행하는 경우 4년에서 5년 정도 소요되는 경우가 많습니다. 학술 분야나 대학의 정책에 따라 다를 수 있으므로, 특정 학교나 프로그램에 대한 정확한 정보는 해당 기관의 공식 웹사이트나 학생 안내서를 참조하는 것이 좋습니다.
[RAG 있음] 통합과정의 재학연한은 8년(16학기)입니다. 

- [RULE_C006]


---
## 8. 검색 품질 실험 — chunk_size

청크가 너무 작으면 조항이 잘리고, 너무 크면 검색이 부정확해질 수 있습니다.

In [ ]:
for size in [150, 280, 450]:
  cs = split_text(rules_text, chunk_size=size)
  idx = [{'chunk_id': f'C{i}', 'text': t} for i, t in enumerate(cs, 1)]
  # 임시 search
  q = tokenize('석사학위과정 수업연한')
  best = max((len(q & tokenize(it['text'])), it['chunk_id']) for it in idx)
  print(f'chunk_size={size}, chunks={len(cs)}, best={best}')

chunk_size=150, chunks=519, best=(2, 'C10')
chunk_size=280, chunks=212, best=(2, 'C5')
chunk_size=450, chunks=120, best=(2, 'C3')
